# Phase 8: Relative Persistence (Rips) — H*(K,L) 단독 계산 및 분류 평가

Six-pack 전체가 아닌, **Relative Persistence Barcode H*(K,L)만** 효율적으로 계산합니다.  
Drel boundary 행렬을 직접 구축하여 (Df 전체를 만들지 않음) 메모리/시간 절약.

## Pipeline
1. 환경 설정 & Google Drive 마운트
2. Rips complex + Relative Barcode 계산
3. PI 벡터화
4. 전 샘플(512) 배치 계산 & `.npz` 저장
5. 벡터 로드 & 분류 평가 (Soft/Strict accuracy, F1)
6. 기존 결과와 비교 (+ **Sixpack_Rips + Relative_Rips 결합 평가**)

**평가 방법: `최종.ipynb` 동일** — Scaling → PCA → **2차 Scaling** → CV

## 1. 환경 설정

In [ ]:
pip install gudhi persim

In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

In [ ]:
import os, time, gc, glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from gudhi import RipsComplex
from collections import defaultdict
from persim import PersistenceImager
import persim.images_weights as weights

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score
from sklearn.base import clone
import psutil
import warnings
warnings.filterwarnings('ignore')

print('All imports loaded.')

## 2. Rips Filtration & 기본 함수

In [ ]:
def compute_Rips(points, max_edge=10):
    """Rips complex 계산 (GUDHI)."""
    rips = RipsComplex(points=points, max_edge_length=max_edge)
    st = rips.create_simplex_tree(max_dimension=2)
    return st


def divide_filtration(st):
    """SimplexTree에서 (simplex, filtration값) 리스트 추출."""
    pairs = [(tuple(sorted(s)), f) for s, f in st.get_filtration()]
    simplices = [p[0] for p in pairs]
    filt     = [p[1] for p in pairs]
    return simplices, filt

## 3. Relative Persistence Barcode 계산

Six-pack 전체를 계산하는 대신, **H*(K,L)에 필요한 Drel 행렬만** 직접 구축합니다.  
Df 전체를 만들지 않으므로 메모리와 시간이 절약됩니다.

In [ ]:
def compute_relative_barcode_only(A, B, max_edge=10):
    """
    Inclusion L = Rips(A) ⊆ K = Rips(A∪B) 에 대한
    Relative Persistence Barcode (H*(K,L))를 효율적으로 계산합니다.
    """
    total = np.concatenate([A, B], axis=0)
    a_len = len(A)

    # 1. Rips complex 계산 및 Filtration 추출
    st = compute_Rips(total, max_edge=max_edge)
    simplices, filt = divide_filtration(st)
    del st, total
    gc.collect()

    # 2. K\L 심플렉스 인덱스 분류
    idx_KmL = [i for i, s in enumerate(simplices) if any(v >= a_len for v in s)]
    set_idx_KmL = set(idx_KmL)
    KmL_pos = {g_idx: l_idx for l_idx, g_idx in enumerate(idx_KmL)}

    # 3. Relative Boundary Matrix 직접 구축 (C*(K)/C*(L))
    sf_to_idx = {s: i for i, s in enumerate(simplices)}
    Drel = []
    for g_idx in idx_KmL:
        s = simplices[g_idx]
        rows = set()
        if len(s) > 1:
            for j in range(len(s)):
                face = s[:j] + s[j+1:]
                face_idx = sf_to_idx.get(face)
                if face_idx in set_idx_KmL:
                    rows.add(KmL_pos[face_idx])
        Drel.append(rows)
    del sf_to_idx, set_idx_KmL, KmL_pos
    gc.collect()

    # 4. 최적화된 행렬 소거 (V 행렬 계산 제외)
    def _reduce_fast(columns):
        m = len(columns)
        R = [set(col) for col in columns]
        low = [-1] * m
        pivot_of_row = {}
        for i in range(m):
            while R[i]:
                li = max(R[i])
                if li in pivot_of_row:
                    R[i] ^= R[pivot_of_row[li]]
                else:
                    pivot_of_row[li] = i
                    low[i] = li
                    break
        return R, low

    Rrel, lowrel = _reduce_fast(Drel)
    del Drel
    gc.collect()

    # 5. Barcode 추출
    def _format(bars_dict):
        out = {}
        for p in [0, 1, 2]:
            if p in bars_dict and bars_dict[p]:
                arr = np.array(bars_dict[p])
                out[p] = arr[np.lexsort((arr[:, 1], arr[:, 0]))]
            else:
                out[p] = np.empty((0, 2))
        return out

    rel_bars = defaultdict(list)
    for pos in range(len(idx_KmL)):
        if lowrel[pos] != -1:
            sigma_local = lowrel[pos]
            sigma = idx_KmL[sigma_local]
            tau = idx_KmL[pos]
            b, d = filt[sigma], filt[tau]
            if abs(b - d) > 1e-12:
                p = len(simplices[sigma]) - 1
                rel_bars[p].append((b, d))

    del Rrel, lowrel, idx_KmL, simplices, filt
    gc.collect()

    return {'relative': _format(rel_bars)}

## 4. Persistence Image 벡터화

In [ ]:
def compute_PIs(barcodes, max_eps=10, px_res=0.1, sigma=0.05, normalization=False):
    """Barcode → Persistence Image 벡터 변환."""
    for key in barcodes:
        if len(barcodes[key]) == 0:
            barcodes[key] = np.zeros((0, 2))

    vector = {}

    # H0
    pi0 = PersistenceImager(pixel_size=px_res,
                            birth_range=(0, 1),
                            pers_range=(0, max_eps),
                            weight=weights.persistence,
                            weight_params={'n': 1},
                            kernel_params={'sigma': [[sigma, 0], [0, sigma]]})
    bars0 = np.array(barcodes[0])
    img0 = pi0.transform(bars0, skew=False) if len(bars0) > 0 else np.zeros((int(1/px_res), int(max_eps/px_res)))
    img0_1d = np.mean(img0, axis=0)
    del img0, pi0

    # H1
    pi1 = PersistenceImager(pixel_size=px_res,
                            birth_range=(0, max_eps),
                            pers_range=(0, max_eps / 2),
                            weight=weights.persistence,
                            weight_params={'n': 1},
                            kernel_params={'sigma': [[sigma, 0], [0, sigma]]})
    bars1 = np.array(barcodes[1])
    img1 = pi1.transform(bars1, skew=True) if len(bars1) > 0 else np.zeros((int(max_eps/px_res), int((max_eps/2)/px_res)))
    del pi1

    if normalization:
        vector[0] = img0_1d / np.max(img0_1d) if np.max(img0_1d) > 0 else img0_1d
        vector[1] = img1.flatten() / np.max(img1) if np.max(img1) > 0 else img1.flatten()
    else:
        vector[0] = img0_1d
        vector[1] = img1.flatten()

    del img1
    return vector

## 5. 배치 계산 (512 샘플)

In [ ]:
BASE_DIR = '/content/drive/MyDrive/URP'
VECTOR_DIR = os.path.join(BASE_DIR, '1224_Vectors')
OUT_DIR = os.path.join(VECTOR_DIR, 'Relative_Rips')
os.makedirs(OUT_DIR, exist_ok=True)

MAX_EDGE = 10   # Rips max edge length
A_VALS = [0.0, 0.01, 0.05, 0.09, 0.13, 0.17, 0.21, 0.25]
PARAM_LIST = [(x1, x2, x3) for x1 in A_VALS for x2 in A_VALS for x3 in A_VALS]
print(f'Total samples: {len(PARAM_LIST)}')
print(f'Output dir: {OUT_DIR}')
print(f'Max edge: {MAX_EDGE}')

In [ ]:
def get_ram_usage_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024

def process_single_sample(idx, params, base_dir, out_dir, max_edge):
    """단일 샘플: Relative H*(K,L) barcode만 계산하여 저장."""
    folder = os.path.join(base_dir, f'ParamSweep_{idx}_Output')
    pos_file = os.path.join(folder, f'Pos_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat')
    types_file = os.path.join(folder, f'Types_{params[0]:.2f}_{params[1]:.2f}_{params[2]:.2f}.dat')

    types = np.loadtxt(types_file, dtype=int)
    positions = np.loadtxt(pos_file, delimiter=',')
    A = positions[types == 1]
    B = positions[types == 2]
    del types, positions

    # Relative barcodes (Rips)
    rel_A2B = compute_relative_barcode_only(A, B, max_edge=max_edge)
    gc.collect()
    rel_B2A = compute_relative_barcode_only(B, A, max_edge=max_edge)
    gc.collect()

    del A, B

    # PI vectorization
    PI_A2B = {}
    for key in list(rel_A2B.keys()):
        PI_A2B[key] = compute_PIs(rel_A2B[key], max_eps=MAX_EDGE*2, normalization=False)
        del rel_A2B[key]
    del rel_A2B

    PI_B2A = {}
    for key in list(rel_B2A.keys()):
        PI_B2A[key] = compute_PIs(rel_B2A[key], max_eps=MAX_EDGE*2, normalization=False)
        del rel_B2A[key]
    del rel_B2A
    gc.collect()

    save_path = os.path.join(out_dir, f'Relative_Rips_{idx}.npz')
    np.savez_compressed(save_path, PI_A2B, PI_B2A)
    del PI_A2B, PI_B2A
    gc.collect()
    return save_path

In [ ]:
# ⚠️ 이미 계산된 파일은 건너뜁니다.

START_IDX = 1
END_IDX = 512

for idx in range(START_IDX, END_IDX + 1):
    save_path = os.path.join(OUT_DIR, f'Relative_Rips_{idx}.npz')
    if os.path.exists(save_path):
        continue

    ram_mb = get_ram_usage_mb()
    print(f'[{idx}] 시작 (RAM: {ram_mb:.0f}MB)')
    start_time = time.time()

    try:
        params = PARAM_LIST[idx - 1]
        saved = process_single_sample(idx, params, BASE_DIR, OUT_DIR, MAX_EDGE)
        elapsed = time.time() - start_time
        ram_mb = get_ram_usage_mb()
        print(f'[{idx}] 완료 ({elapsed:.1f}초, RAM: {ram_mb:.0f}MB)')

    except Exception as e:
        print(f'[{idx}] 에러: {e}')

    gc.collect()

print('\n배치 계산 완료!')

## 6. Ground Truth & 평가 함수

**`최종.ipynb`와 동일한 방식** — 실제 phase diagram에서 추출한 ADJACENT_PHASES 사용

In [ ]:
# Ground Truth Matrix
M1=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3]]
M2=[[0,0,1,1,1,1,1,1],[0,0,1,1,1,1,1,1],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,3,4],[2,2,3,3,3,3,3,3],[2,2,3,3,3,3,4,4],[2,2,3,3,3,3,3,3]]
M3=[[6,6,7,7,7,7,7,7],[6,6,6,7,7,7,7,7],[9,6,3,3,3,3,3,3],[9,10,3,4,4,3,3,4],[9,10,3,3,4,4,3,4],[9,10,3,4,4,4,4,4],[9,10,3,4,3,4,4,4],[9,10,3,4,3,4,4,4]]
M4=[[6,6,12,12,7,7,7,7],[6,6,12,12,7,7,7,7],[9,6,6,11,7,7,4,4],[9,9,6,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,3,3,4,4,4],[9,9,10,4,4,4,4,4],[9,9,10,4,4,4,4,4]]
M5=[[6,6,12,12,12,12,7,7],[6,6,12,12,12,12,12,7],[9,9,6,11,11,11,12,11],[9,9,6,11,11,11,4,4],[9,9,13,13,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,13,10,4,4,4,4],[9,9,10,10,4,4,4,4]]
M6=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,11],[9,9,6,11,11,11,11,11],[9,9,6,6,6,13,4,4],[9,9,6,13,13,4,4,4],[9,9,6,13,4,4,4,4],[9,9,6,13,4,4,4,4]]
M7=[[6,6,12,12,12,12,12,12],[9,6,12,12,12,12,12,12],[9,6,6,11,11,11,11,12],[9,6,6,11,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,11,11,11,4],[9,9,6,6,13,13,4,4],[9,9,6,13,13,4,4,4]]
M8=[[6,12,12,12,12,12,12,12],[6,6,12,12,12,12,12,12],[9,6,6,6,11,11,11,11],[9,6,6,6,11,11,11,11],[9,9,6,6,11,11,11,11],[9,9,6,6,6,11,11,11],[9,9,6,6,13,13,11,11],[9,9,6,6,13,13,11,4]]
GROUND_TRUTH_M = np.asarray([M1, M2, M3, M4, M5, M6, M7, M8])

def get_label_from_index(task_id):
    """
    Multi_3.ipynb의 task_id로부터 GT 값 계산
    task_id = 1 + RR_idx*64 + RG_idx*8 + GG_idx
    GROUND_TRUTH_M[RG_idx][RR_idx][GG_idx]로 접근
    """
    idx = task_id - 1
    RR_idx = idx // 64
    RG_idx = (idx % 64) // 8
    GG_idx = idx % 8
    return GROUND_TRUTH_M[RG_idx][RR_idx][GG_idx]

# ADJACENT_PHASES (실제 phase diagram 기반, 최종.ipynb와 동일)
ADJACENT_PHASES = {
    0: [1, 2],
    1: [0, 3],
    2: [0, 3],
    3: [1, 2, 4, 6, 7, 10, 11],
    4: [3, 7, 10, 11, 12, 13],
    6: [3, 7, 9, 10, 11, 12, 13],
    7: [3, 4, 6, 11, 12],
    9: [6, 10, 13],
    10: [3, 4, 6, 9, 13],
    11: [3, 4, 6, 7, 12, 13],
    12: [4, 6, 7, 11],
    13: [4, 6, 9, 10, 11],
}

def are_adjacent_phases(label1, label2):
    """두 레이블이 인접한 위상인지 확인"""
    label1, label2 = int(label1), int(label2)
    if label1 == label2:
        return True
    if label1 in ADJACENT_PHASES and label2 in ADJACENT_PHASES[label1]:
        return True
    if label2 in ADJACENT_PHASES and label1 in ADJACENT_PHASES[label2]:
        return True
    return False

def soft_accuracy_score(y_true, y_pred, ignore_adjacent=True):
    """
    인접 위상 간의 오분류를 무시하는 정확도 계산

    논문: "we modified the scoring function to ignore misclassification
    between adjacent phases in the accuracy computation"
    """
    if not ignore_adjacent:
        return accuracy_score(y_true, y_pred)

    n_correct = 0
    n_total = len(y_true)

    for true_label, pred_label in zip(y_true, y_pred):
        if true_label == pred_label:
            n_correct += 1
        elif are_adjacent_phases(true_label, pred_label):
            n_correct += 1

    return n_correct / n_total if n_total > 0 else 0.0

print(f'Classes: {np.unique(GROUND_TRUTH_M)} ({len(np.unique(GROUND_TRUTH_M))} classes)')
print(f'ADJACENT_PHASES keys: {sorted(ADJACENT_PHASES.keys())}')

## 7. 벡터 로드

In [ ]:
def load_npz_data(data_dir, prefix):
    """범용 NPZ 데이터 로더. {prefix}_{idx}.npz 형식 파일을 로드합니다.
    Returns: (X, y, indices) 또는 데이터 없으면 (None, None, None)
    """
    npz_files = sorted(glob.glob(os.path.join(data_dir, f'{prefix}_*.npz')))
    if not npz_files:
        return None, None, None
    X_list, y_list, indices = [], [], []
    for filepath in npz_files:
        filename = os.path.basename(filepath)
        try:
            sim_idx = int(filename.split('_')[-1].split('.')[0])
            label = get_label_from_index(sim_idx)
            data = np.load(filepath, allow_pickle=True)
            arr_0 = data['arr_0'].item() if hasattr(data['arr_0'], 'item') else data['arr_0']
            arr_1 = data['arr_1'].item() if hasattr(data['arr_1'], 'item') else data['arr_1']
            features = []
            for arr in [arr_0, arr_1]:
                if isinstance(arr, dict):
                    for key in sorted(arr.keys()):
                        val = arr[key]
                        if isinstance(val, dict):
                            for dk in sorted(val.keys()):
                                features.extend(np.array(val[dk]).flatten())
                        else:
                            features.extend(np.array(val).flatten())
                else:
                    features.extend(np.array(arr).flatten())
            X_list.append(features)
            y_list.append(label)
            indices.append(sim_idx)
        except Exception as e:
            print(f'  Error: {filename} - {e}')
    if not X_list:
        return None, None, None
    return np.nan_to_num(np.array(X_list)), np.array(y_list), indices

# Relative_Rips 로드
X_rel, y_rel, idx_rel = load_npz_data(OUT_DIR, 'Relative_Rips')
if X_rel is not None:
    print(f'Relative_Rips: X.shape={X_rel.shape}, y.shape={y_rel.shape}')
else:
    print('Relative_Rips: No data found. Run batch computation first.')

## 8. 분류 평가 (5-fold Stratified CV)

**`최종.ipynb` 동일 방식 적용:**
1. `StandardScaler` → PCA
2. **PCA 후 2차 `StandardScaler`** (각 fold 내 train/test 분리)
3. Soft-SVM C값 sweep 포함

In [ ]:
# ============================================
# CONFIGURATION (최종.ipynb 동일)
# ============================================
C_VALUES = [0.5, 1.0, 2.0]      # Soft margin C parameters
REDUCTION_DIM = 20              # PCA target dimension
N_SPLITS = 5                    # Cross-validation folds
RANDOM_STATE = 42

def evaluate_classifiers(X, y, pca_dim=20, n_splits=5):
    """
    최종.ipynb와 동일한 평가 파이프라인:
    1) StandardScaler → PCA (전체 데이터에 대해)
    2) CV 루프 내에서 Soft-SVM은 내부 StandardScaler 적용 (2차 스케일링)
    3) 일반 분류기는 PCA 출력에 직접 적용
    """
    # 1차 스케일링 + PCA (최종.ipynb: main execution block)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    original_dim = X_scaled.shape[1]

    if pca_dim and X_scaled.shape[1] > pca_dim:
        pca = PCA(n_components=pca_dim, random_state=RANDOM_STATE)
        X_reduced = pca.fit_transform(X_scaled)
        print(f'  PCA: {original_dim}D -> {pca_dim}D')
    else:
        X_reduced = X_scaled
        print(f'  No PCA needed (dim={original_dim})')

    # ── Soft Margin SVM 평가 (최종.ipynb: evaluate_soft_margin_svm) ──
    # 핵심: 내부에서 2차 StandardScaler 적용
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    results = {}

    for C in C_VALUES:
        soft_a, strict_a, f1_s = [], [], []
        # 2차 스케일링 (최종.ipynb evaluate_soft_margin_svm 내부)
        scaler2 = StandardScaler()
        X_scaled2 = scaler2.fit_transform(X_reduced)

        for tr, te in skf.split(X_scaled2, y):
            clf = SVC(kernel='rbf', C=C, gamma='scale', random_state=RANDOM_STATE)
            clf.fit(X_scaled2[tr], y[tr])
            yp = clf.predict(X_scaled2[te])
            soft_a.append(soft_accuracy_score(y[te], yp, ignore_adjacent=True))
            strict_a.append(accuracy_score(y[te], yp))
            f1_s.append(f1_score(y[te], yp, average='weighted', zero_division=0))
        results[f'Soft-SVM (C={C})'] = {
            'soft': (np.mean(soft_a)*100, np.std(soft_a)*100),
            'strict': (np.mean(strict_a)*100, np.std(strict_a)*100),
            'f1': (np.mean(f1_s)*100, np.std(f1_s)*100),
        }

    # ── 일반 분류기 평가 (최종.ipynb: evaluate_all_classifiers) ──
    # PCA 출력에 직접 적용 (2차 스케일링 없음)
    clfs = {
        'KNN (k=3)': KNeighborsClassifier(n_neighbors=3),
        'KNN (k=12)': KNeighborsClassifier(n_neighbors=12),
        'SVM (RBF)': SVC(kernel='rbf', C=1.0),
        'SVM (Linear)': SVC(kernel='linear', C=1.0),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
    }

    for name, tmpl in clfs.items():
        soft_a, strict_a, f1_s = [], [], []
        for tr, te in skf.split(X_reduced, y):
            c = clone(tmpl)
            c.fit(X_reduced[tr], y[tr])
            yp = c.predict(X_reduced[te])
            soft_a.append(soft_accuracy_score(y[te], yp, ignore_adjacent=True))
            strict_a.append(accuracy_score(y[te], yp))
            f1_s.append(f1_score(y[te], yp, average='weighted', zero_division=0))
        results[name] = {
            'soft': (np.mean(soft_a)*100, np.std(soft_a)*100),
            'strict': (np.mean(strict_a)*100, np.std(strict_a)*100),
            'f1': (np.mean(f1_s)*100, np.std(f1_s)*100),
        }

    return results

if X_rel is not None:
    for pca_d in [20, 50, None]:
        label = f'PCA={pca_d}D' if pca_d else 'No PCA'
        print(f'\n{"="*70}')
        print(f'Relative_Rips  {label}, 5-fold CV')
        print(f'{"="*70}')
        res = evaluate_classifiers(X_rel, y_rel, pca_dim=pca_d)
        for n, r in res.items():
            print(f'  {n:25s} | Soft: {r["soft"][0]:.2f}+/-{r["soft"][1]:.2f}%  '
                  f'| Strict: {r["strict"][0]:.2f}+/-{r["strict"][1]:.2f}%  '
                  f'| F1: {r["f1"][0]:.2f}+/-{r["f1"][1]:.2f}%')

## 9. Sixpack_Rips + Relative_Rips 결합 평가

기존 **Sixpack_Rips** 벡터에 새로 계산한 **Relative_Rips** 벡터를 **concatenate**하여  
결합된 특징벡터가 분류 성능을 향상시키는지 평가합니다.

### 결합 방식
- 동일한 sample index를 가진 파일끼리 매칭
- 각 샘플의 feature vector를 `np.concatenate([sixpack_rips_feat, relative_rips_feat])`
- 한쪽만 있는 샘플은 제외

In [ ]:
# Sixpack_Rips 로드
RIPS_DIR = os.path.join(VECTOR_DIR, 'Sixpack_Rips')
X_rips, y_rips, idx_rips = load_npz_data(RIPS_DIR, 'Sixpack_Rips')

if X_rips is not None:
    print(f'Sixpack_Rips:  X.shape={X_rips.shape}, y.shape={y_rips.shape}')
else:
    print('Sixpack_Rips: No data found.')

# ── 결합 ──────────────────────────────────────────────────────────
X_combined, y_combined = None, None

if X_rips is not None and X_rel is not None:
    # 공통 sample index 찾기
    rips_map = {sid: i for i, sid in enumerate(idx_rips)}
    rel_map  = {sid: i for i, sid in enumerate(idx_rel)}
    common_idx = sorted(set(idx_rips) & set(idx_rel))

    if common_idx:
        X_comb_list = []
        y_comb_list = []
        for sid in common_idx:
            feat_rips = X_rips[rips_map[sid]]
            feat_rel  = X_rel[rel_map[sid]]
            X_comb_list.append(np.concatenate([feat_rips, feat_rel]))
            y_comb_list.append(y_rips[rips_map[sid]])

        X_combined = np.array(X_comb_list)
        y_combined = np.array(y_comb_list)

        print(f'\nSixpack_Rips + Relative_Rips 결합:')
        print(f'  공통 샘플 수: {len(common_idx)}')
        print(f'  Sixpack_Rips dim:  {X_rips.shape[1]}')
        print(f'  Relative_Rips dim: {X_rel.shape[1]}')
        print(f'  결합 벡터 dim:     {X_combined.shape[1]}  ({X_rips.shape[1]} + {X_rel.shape[1]})')
    else:
        print('\n공통 샘플이 없어 결합할 수 없습니다.')

In [ ]:
# 결합 벡터 분류 평가
if X_combined is not None:
    for pca_d in [20, 50, None]:
        label = f'PCA={pca_d}D' if pca_d else 'No PCA'
        print(f'\n{"="*70}')
        print(f'Sixpack_Rips + Relative_Rips (결합)  {label}, 5-fold CV')
        print(f'{"="*70}')
        res = evaluate_classifiers(X_combined, y_combined, pca_dim=pca_d)
        for n, r in res.items():
            print(f'  {n:25s} | Soft: {r["soft"][0]:.2f}+/-{r["soft"][1]:.2f}%  '
                  f'| Strict: {r["strict"][0]:.2f}+/-{r["strict"][1]:.2f}%  '
                  f'| F1: {r["f1"][0]:.2f}+/-{r["f1"][1]:.2f}%')
else:
    print('결합 벡터 없음 -- Sixpack_Rips 또는 Relative_Rips 데이터가 필요합니다.')

## 10. 전체 비교 요약

| Method | 설명 |
|:-------|:-----|
| Relative_Rips | H*(K,L)만 사용 |
| Sixpack_Rips | Image/Kernel/Cokernel/Relative + Ordinary 사용 |
| Sixpack_Cech | Cech 기반 Six-pack |
| **Sixpack_Rips + Relative_Rips** | Six-pack에 독립 계산된 Relative barcode 추가 결합 |

In [ ]:
comparison = {}

# 1) Relative_Rips (단독)
if X_rel is not None:
    res = evaluate_classifiers(X_rel, y_rel, pca_dim=20)
    best = max(res, key=lambda k: res[k]['soft'][0])
    comparison['Relative_Rips'] = {**res[best], 'best_clf': best}

# 2) Sixpack_Rips (단독)
if X_rips is not None:
    res = evaluate_classifiers(X_rips, y_rips, pca_dim=20)
    best = max(res, key=lambda k: res[k]['soft'][0])
    comparison['Sixpack_Rips'] = {**res[best], 'best_clf': best}

# 3) Sixpack_Rips + Relative_Rips (결합)
if X_combined is not None:
    res = evaluate_classifiers(X_combined, y_combined, pca_dim=20)
    best = max(res, key=lambda k: res[k]['soft'][0])
    comparison['Sixpack_Rips + Rel_Rips'] = {**res[best], 'best_clf': best}

# 4) Sixpack_Cech (있으면)
CECH_DIR = os.path.join(VECTOR_DIR, 'Sixpack_Cech')
X_cech, y_cech, _ = load_npz_data(CECH_DIR, 'Sixpack_Cech')
if X_cech is not None:
    res = evaluate_classifiers(X_cech, y_cech, pca_dim=20)
    best = max(res, key=lambda k: res[k]['soft'][0])
    comparison['Sixpack_Cech'] = {**res[best], 'best_clf': best}

# ── 결과 출력 ──────────────────────────────────────────────────────
if comparison:
    print(f'\n{"="*100}')
    print(f'{"Method":30s} | {"Best Classifier":25s} | {"Soft Acc":15s} | {"Strict Acc":15s} | {"F1":15s}')
    print(f'{"="*100}')
    for method, r in comparison.items():
        print(f'{method:30s} | {r["best_clf"]:25s} | '
              f'{r["soft"][0]:5.2f}+/-{r["soft"][1]:.2f}%  | '
              f'{r["strict"][0]:5.2f}+/-{r["strict"][1]:.2f}%  | '
              f'{r["f1"][0]:5.2f}+/-{r["f1"][1]:.2f}%')
    print(f'{"="*100}')
else:
    print('No comparison data available.')

## 11. Sixpack_Rips vs Sixpack_Rips + Relative_Rips 상세 비교

기존 Sixpack_Rips 단독 성능과 Relative_Rips를 결합한 성능을 **분류기별로** 비교합니다.  
Δ 열은 결합으로 인한 성능 변화량(양수=향상)을 나타냅니다.

In [ ]:
if X_rips is not None and X_combined is not None:
    pca_d = 20
    res_rips = evaluate_classifiers(X_rips, y_rips, pca_dim=pca_d)
    res_comb = evaluate_classifiers(X_combined, y_combined, pca_dim=pca_d)

    print(f'PCA={pca_d}D, 5-fold Stratified CV')
    print(f'Sixpack_Rips dim: {X_rips.shape[1]}  |  결합 dim: {X_combined.shape[1]}')
    print()

    header = (f'{"Classifier":25s} | {"Soft(기존)":>12s} | '
              f'{"Soft(결합)":>12s} | {"d Soft":>8s} | '
              f'{"Strict(기존)":>12s} | {"Strict(결합)":>12s} | {"d Strict":>8s}')
    sep = '-' * len(header)
    print(sep)
    print(header)
    print(sep)

    for clf_name in res_rips:
        sr = res_rips[clf_name]
        cr = res_comb[clf_name]
        d_soft = cr['soft'][0] - sr['soft'][0]
        d_strict = cr['strict'][0] - sr['strict'][0]
        sign_s = '+' if d_soft >= 0 else ''
        sign_st = '+' if d_strict >= 0 else ''

        print(f'{clf_name:25s} | '
              f'{sr["soft"][0]:5.2f}+/-{sr["soft"][1]:.1f}% | '
              f'{cr["soft"][0]:5.2f}+/-{cr["soft"][1]:.1f}% | '
              f'{sign_s}{d_soft:5.2f}%p | '
              f'{sr["strict"][0]:5.2f}+/-{sr["strict"][1]:.1f}% | '
              f'{cr["strict"][0]:5.2f}+/-{cr["strict"][1]:.1f}% | '
              f'{sign_st}{d_strict:5.2f}%p')

    print(sep)

    # 최고 성능 요약
    best_rips = max(res_rips, key=lambda k: res_rips[k]['soft'][0])
    best_comb = max(res_comb, key=lambda k: res_comb[k]['soft'][0])
    print(f'\n  * Sixpack_Rips 최고:         {best_rips} -> Soft {res_rips[best_rips]["soft"][0]:.2f}%')
    print(f'  * Sixpack_Rips+Rel_Rips 최고: {best_comb} -> Soft {res_comb[best_comb]["soft"][0]:.2f}%')
    delta = res_comb[best_comb]['soft'][0] - res_rips[best_rips]['soft'][0]
    sign = '+' if delta >= 0 else ''
    print(f'  -> 최고 성능 변화: {sign}{delta:.2f}%p')
else:
    print('비교 불가 -- Sixpack_Rips 또는 결합 데이터가 필요합니다.')

## 완료

Phase 8: Relative Persistence (Rips only) 실험 완료.  
Sixpack_Rips + Relative_Rips 결합이 분류 성능을 향상시키는지 확인합니다.